In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from src.imputation import imputation_normal_distribution, log2
from src.correlation import pairwise_correlation
from src.data_processing import pre_processing, calculate_data_completeness, filter_data, summarize_filtered_data
from src.data_processing import compute_pca,generate_pca_plot, calculate_cv
from src.statistical_testing import perform_linear_regression
import pingouin as pg
from matplotlib_venn import venn3, venn3_circles
from venn import venn
import statsmodels.stats.multitest as multi
from tqdm import tqdm
from scipy.stats import pearsonr
from scipy.stats import zscore
import pickle

### Proteomics data processing

#### Define directories

In [ ]:
import os
from pathlib import Path

# Get the number of available CPUs
CPUS = os.cpu_count()

# Define the paths for the raw and processed data folders
DATA_FOLDER_1k = Path('/Volumes/auditing-groupdirs/SUN-CPR-TARGET_PROTEOMICS/1k_replication')
DATA_FOLDER_RAW = Path(os.path.join(DATA_FOLDER_1k, 'data/raw'))
DATA_FOLDER_PROCESSED = Path(os.path.join(DATA_FOLDER_1k, 'data/processed'))
DATA_FOLDER_CLINIC = '/Volumes/auditing-groupdirs/SUN-CBMR-Childhood-Genetic-TCOC/Proteomics analysis/GitHub/TARGET/batch2023'
pQTL_2k_FOLDER = Path('/Volumes/auditing-groupdirs/SUN-CPR-TARGET_PROTEOMICS/2k_discovery/pQTL/')

# Ensure base folders are created and define subfolder paths
os.makedirs(DATA_FOLDER_PROCESSED, exist_ok=True)
subfolders = ['tables', 'results', 'figures', 'pQTL', 'dash', 'annotations']
folders = {f: Path(DATA_FOLDER_1k, f) for f in subfolders}

# Create subfolders if they don't exist
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)
    
Path(folders['pQTL'] / 'gemma').mkdir(exist_ok=True)
gemma_path = Path(folders['pQTL'], 'gemma')

#### Import data and format column headers

In [ ]:
# Read annotation file
annotation_file = pd.read_csv(os.path.join(DATA_FOLDER_RAW, 'anno_test.csv'), sep=';')

# Get the sample IDs to use for the analysis
non_qa_samples = annotation_file[annotation_file['Grouping_plate'] != 'QA']
sample_ids = non_qa_samples['Sample ID'].tolist()
IDmapping_sampleID_to_Batch = dict(zip(annotation_file['Sample ID'], annotation_file['Grouping_batch']))

In [ ]:
report_filename = '20240301_214956_Protein Lili long (Normal)_1k_canonical.tsv'
cols_to_keep = ['R.FileName', 'PG.Genes', 'PG.ProteinAccessions', 'PG.Quantity']

RE_READ = False
if not RE_READ:
    data_raw_long = pd.read_pickle(os.path.join(DATA_FOLDER_PROCESSED, 'data_raw_long.pkl'))
    
else:   
    chunk_size = 10000
    file_to_read = pd.read_csv(os.path.join(DATA_FOLDER_RAW, report_filename), 
                               delimiter='\t', na_values='Filtered', usecols=cols_to_keep, chunksize=chunk_size)
    chunks = []
    for chunk in tqdm(file_to_read):
        chunks.append(chunk)
    report_plasma = pd.concat(chunks)
    data_raw_long = report_plasma.dropna().reset_index().drop(['index'], axis=1)
    data_raw_long.to_pickle(os.path.join(DATA_FOLDER_PROCESSED, 'data_raw_long.pkl'))

In [ ]:
data_raw_long = pre_processing(data_raw_long)
protein_ids = data_raw_long[['Gene names', 'Protein IDs', 
                             'Gene name', 'Protein ID',
                             'ProteinID_Genename']].drop_duplicates()

In [ ]:
cond1 = data_raw_long['Sample ID']=='PlateR1_1'
cond2 = data_raw_long['Protein ID']=='Q9Y646'
test_value=data_raw_long[(cond1) & (cond2)]['PG.Quantity'].iloc[0]
assert abs(test_value-75.882980)<1e-6, 'Value changed compared to previous run'

In [ ]:
#Prepare ID list for genome coordinate mapping
protein_ids['Protein IDs_list'] = protein_ids['Protein IDs'].str.split(';')
protein_ids_all = protein_ids.explode('Protein IDs_list')

with open(gemma_path / 'protein_id_all.list', 'w') as file:
    for protein in protein_ids_all['Protein IDs_list']:
        file.write(str(protein) + '\n')
        
protein_ids_all.to_csv(gemma_path / 'protein_id_all.txt', index=False, sep='\t')

In [ ]:
cols_to_keep = ['Sample ID', 'ProteinID_Genename', 'PG.Quantity']
data_plasma_raw = data_raw_long[cols_to_keep].pivot(columns='Sample ID', index='ProteinID_Genename', values='PG.Quantity')

In [ ]:
df_raw_comp = calculate_data_completeness(data_plasma_raw)
sns.lineplot(x='rank', y='%Complete', data=df_raw_comp)

#### Filter data based on data completeness

In [ ]:
proteins, sample_ids, data_plasma_filtered = filter_data(data_plasma_raw)

#### Summarize filtered data

In [ ]:
summarize_filtered_data(data_plasma_filtered)

#### Plot proteins by data completeness

In [ ]:
df_filtered_comp = calculate_data_completeness(data_plasma_filtered)
sns.lineplot(x='rank', y='%Complete', data=df_filtered_comp)
plt.ylim(0, 1.1)

In [ ]:
data_plasma_filtered_log = data_plasma_filtered.apply(log2)

#### Generate dataframe to match between protein IDs in the same PG

In [ ]:
IDmapping_proteinid_to_proteinids = dict(zip(protein_ids['Protein ID'], protein_ids['Protein IDs']))
IDmapping_genename_to_genenames = dict(zip(protein_ids['Gene name'], protein_ids['Gene names']))
IDmapping_proteinid_to_genename = dict(zip(protein_ids['Protein ID'], protein_ids['Gene name']))

#df_sig_dis = pd.read_csv(pQTL_2k_FOLDER / 'df_sig_primary_upgrade.csv')
df_sig_dis = pd.read_csv(pQTL_2k_FOLDER / 'df_sig_primary_final.csv')
df_proteinID_mapping = protein_ids_all[protein_ids_all['Protein IDs_list'].isin(df_sig_dis['Protein ID'].unique())]
df_proteinID_mapping.to_csv(folders['pQTL'] / 'ProteinID_Mapping.csv', sep='\t', index=False)

#### Compute PCA

In [ ]:
X_train = data_plasma_filtered_log.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]

In [ ]:
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')

#### PCA plot

In [ ]:
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2, )
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

In [ ]:
qc_markers_platelet = pd.read_excel(os.path.join(folders['annotations'], 'plasma_quality_markers.xlsx'), 
                                     sheet_name='platelet', engine='openpyxl')['Gene names'].tolist()
df_loadings['platelet_marker'] = np.where(df_loadings['Gene name'].isin(qc_markers_platelet), 1, 0)

In [ ]:
px.scatter(data_frame=df_loadings, x='PC1', y='PC2', hover_name=df_loadings.index, color='platelet_marker')

### Imputation

In [ ]:
data_plasma_filtered_log_imputed = data_plasma_filtered_log.apply(imputation_normal_distribution)
value = data_plasma_filtered_log_imputed.loc['Q9Y624_F11R', 'PlateR1_2']
assert abs(value - 7.865703) < 1e-6, 'Imputed value changed in comparison to previous run'

In [ ]:
X_train = data_plasma_filtered_log_imputed.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

### Normalization

In [ ]:
from combat.pycombat import pycombat
dm = data_plasma_filtered_log_imputed.copy()
batch = [IDmapping_sampleID_to_Batch[i] for i in dm.columns]
data_plasma_filtered_log_imputed_corrected = pycombat(dm, batch)

In [ ]:
value = data_plasma_filtered_log_imputed_corrected.loc['Q9Y624_F11R', 'PlateR1_2']
assert abs(value - 7.896046) < 1e-6, 'Corrected value changed in comparison to previous run'

In [ ]:
X_train = data_plasma_filtered_log_imputed_corrected.T.dropna(axis=1)
pca, df_pc, df_loadings = compute_pca(dataframe=X_train)
df_loadings['Gene name'] = df_loadings.index.str.split('_').str[1]
df_pc = df_pc.join(annotation_file.set_index('Sample ID'), how='left')
fig_pca = generate_pca_plot(df_pc=df_pc, pca=pca, PCA_x=1, PCA_y=2)
#fig_pca.savefig(os.path.join(folders['figures'], 'PCA.pdf'), bbox_inches='tight', dpi=120)

In [ ]:
df_loadings['platelet_marker'] = np.where(df_loadings['Gene name'].isin(qc_markers_platelet), 1, 0)
px.scatter(data_frame=df_loadings, x='PC1', y='PC2', hover_name=df_loadings.index, color='platelet_marker')

### Quality assessment

#### Calculate CV based on 44 quality assessment samples (pooled plasma allocated in 11 plates)

In [ ]:
qa_plasma = annotation_file[annotation_file['Grouping_batch'] == 'QA']['Sample ID'].tolist()
df_cv = calculate_cv(data_plasma_filtered, qa_samples=qa_plasma).sort_values(by='Coefficient of variation')
df_cv['Gene name'] = df_cv.index.str.split('_').str[1]

#### Depth

In [ ]:
prot_dep_wide = pd.DataFrame({'raw': data_plasma_raw.count(), 'filtered':data_plasma_filtered.count()})
prot_dep = pd.melt(prot_dep_wide, var_name='dataset', value_name='Number of proteins')
prot_dep.groupby('dataset')['Number of proteins'].median()

#### Supplementary figure 1

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12,4))
sns.boxplot(data = prot_dep, x='dataset', y='Number of proteins', color='white', ax=ax1)
ax1.set_ylim(0, 1800)
for i,box in enumerate(ax1.artists):
    box.set_edgecolor('black')
    box.set_facecolor('white')

    # iterate over whiskers and median lines
    for j in range(6*i,6*(i+1)):
         ax1.lines[j].set_color('black')
ax2 = sns.scatterplot(x=df_cv['rank'], y=df_cv['Protein abundance [Log10]'], ax=ax2)
ax3 = sns.scatterplot(x=df_cv['Protein abundance [Log2]'], y=df_cv['Coefficient of variation'], hue=df_cv['color'], ax=ax3)
plt.rcParams['pdf.fonttype'] = 42
#plt.savefig('1k_replication/figures/data_quality.pdf', dpi=120, bbox_inches='tight')

### Double key between genomics and proteomics data

In [ ]:
# Drop rows with all missing values
double_id_file = pd.read_csv(DATA_FOLDER_RAW / 'holbaek_ms_proteomics_batch2023_plasma.csv', sep=';', index_col='blood_sample_ID')
double_id_file = double_id_file.dropna(how='all')

# Create a dictionary to map blood sample IDs to proteomics sample IDs and vice versa
IDmapping_bloodSampleID_to_SampleID = double_id_file['Sample ID'].to_dict()
IDmapping_SampleID_to_bloodSampleID = dict(zip(double_id_file['Sample ID'], double_id_file.index))

In [ ]:
# Define the path to the output file
Path(os.path.join(folders['pQTL'], 'phenomics')).mkdir(exist_ok=True)
output_file_path = os.path.join(folders['pQTL'], 'phenomics', 'IDmapping_SampleID_to_bloodSampleID.p')

# Dump the dictionary to the output file
with open(output_file_path, 'wb') as output_file:
    pickle.dump(IDmapping_SampleID_to_bloodSampleID, output_file)

### Clinical data wrangling

#### Import data and add relevant columns

In [ ]:
DATA_FOLDER_CLINIC

In [ ]:
# Read in the raw clinical data file
dtype_dict = {'z_BMI.Nysom': np.float64}
data_cli_raw = pd.read_csv(os.path.join(DATA_FOLDER_CLINIC, 'HOL_dataset_massspec_bas_clean_v2023.08.01.csv'), 
                          low_memory=False, dtype=dtype_dict)

# Drop unnecessary columns and rows with all missing values
data_cli_raw.dropna(how='all', axis=1, inplace=True)

# Rename columns and add a proteomics analysis date column
data_cli_raw.rename({'gender':'sex'}, axis=1, inplace=True)
data_cli_raw['proteomics_analysis_date'] = pd.Timestamp('2021-08-04')

# Assign groups of overweight and normalweight 
data_cli_raw['obesity'] = np.where(data_cli_raw['z_BMI.Nysom']>=1.28, 1, 0)

# Add a column for the time between blood sample collection and analysis
#data_cli_raw['blood_sample_date_ts']=[pd.Timestamp(s) for s in data_cli_raw['blood_sample_date']]
data_cli_raw['blood_sample_date_ts'] = pd.to_datetime(data_cli_raw['blood_sample_date'])
storagetime = data_cli_raw['proteomics_analysis_date'] - data_cli_raw['blood_sample_date_ts']
data_cli_raw['time_to_analysis'] = storagetime.dt.days

# Drop any duplicated rows and save the filtered DataFrame to a variable
data_cli_filtered = data_cli_raw[~data_cli_raw.index.duplicated(keep=False)]

print('Filtered clinical data shape: {}'.format(data_cli_filtered.shape))

#### Extract clinical data for samples with proteomics measurement

In [ ]:
data_cli_prot = pd.DataFrame(data= data_plasma_raw.T.index, columns=['Sample ID']).set_index('Sample ID')
data_cli_prot['blood_sample_ID'] = data_cli_prot.index.map(IDmapping_SampleID_to_bloodSampleID)
data_cli_prot = data_cli_prot.reset_index().merge(data_cli_filtered, how='left', left_on='blood_sample_ID', 
                                                  right_on='biobank_ID').set_index('Sample ID')
data_cli_prot.drop('blood_sample_ID', axis=1, inplace=True)

data_cli_prot['age_int']=data_cli_prot['age'].round(0)
bin_numbers_bmi = pd.qcut(
    x=data_cli_prot['BMI'], q=20, labels=False, duplicates='drop'
)
data_cli_prot['bin_numbers_bmi'] = bin_numbers_bmi

In [ ]:
dates = [i for i in data_cli_prot['visit_date'] if type(i)==str]
dates_year = [int(i.split('-')[0]) for i in dates]
data_cli_prot['visit_date_year'] = data_cli_prot['visit_date'].map(dict(zip(dates, dates_year)))
data_cli_prot['visit_date_year_binary'] = np.where(data_cli_prot['visit_date_year']>2015, '>2015', '≤2015')

df = data_cli_prot.copy()
df['mr_liverfat_above5%'] = np.where(df['mr_real_liverfat_percent']>5, 1, 0)
df['mr_liverfat_above1.5%'] = np.where(df['mr_real_liverfat_percent']>1.5, 1, 0)
df.loc[df['mr_real_liverfat_percent'].isnull(), 'mr_liverfat_above5%'] = np.nan
df.loc[df['mr_real_liverfat_percent'].isnull(), 'mr_liverfat_above1.5%'] = np.nan

prot_batch = pd.get_dummies(annotation_file.set_index('Sample ID')['Grouping_batch'])
prot_batches = prot_batch.columns.tolist()
df = df.join(prot_batch)
data_cli_prot = df.copy()

In [ ]:
# Save clinical data for phenomics analysis 
data_cli_prot.reset_index().to_csv(folders['pQTL'] / 'phenomics/data_cli_prot.csv', index=False)

In [ ]:
data_cli_prot['sex'].value_counts(1)

### Export data for Dash app

In [ ]:
data_dash = data_plasma_filtered_log.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)[proteins + ['age_int', 'sex', 'z_BMI.Nysom']]
Path(os.path.join(folders['dash'], 'dataset')).mkdir(exist_ok=True)
data_dash.to_csv(os.path.join(folders['dash'], 'dataset/data_age_sex.csv'))

### Baseline participant characteristics

In [ ]:
data_cli_base = data_cli_prot.copy()
data_cli_base = data_cli_base[data_cli_base['QA']==0]

para_toinclude = ['age_year', 'sex', 'z_BMI.Nysom','BMI','height', 'weight', 
                  'tanner_stage', 'pubertal_status2', 'triglycerides', 
                  'chol_total', 'chol_ldl', 'chol_hdl',
                  'glucose', 'insulin', 'HbA1c', 'ALAT', 'ASAT', 'GGT', 'obesity']

data_cli_base = data_cli_base[para_toinclude]

In [ ]:
data_cli_base.groupby('obesity')['pubertal_status2'].value_counts(1)

In [ ]:
from scipy.stats import ttest_ind
cond1 = data_cli_base['obesity']==1
array1 = data_cli_base[(data_cli_base['sex']==1) & (cond1)]['z_BMI.Nysom']
array2 = data_cli_base[(data_cli_base['sex']==2) & (cond1)]['z_BMI.Nysom']
t_statistic, p_value = ttest_ind(array1, array2)
p_value

In [ ]:
median = data_cli_base.groupby('obesity').median().T
median.columns = [str(i) + '_median' for i in median.columns]
q1 = data_cli_base.groupby('obesity').quantile(0.25).T
q1.columns = [str(i)+ '_q1' for i in q1.columns]
q3 = data_cli_base.groupby('obesity').quantile(0.75).T
q3.columns = [str(i)+ '_q3' for i in q3.columns]
df_median = pd.concat([median, q1, q3], axis=1)[['0.0_median', '0.0_q1', '0.0_q3', '1.0_median', '1.0_q1', '1.0_q3']]

In [ ]:
data_cli_base.groupby('obesity')['pubertal_status2'].value_counts(1)

### Combine proteomics and clinical data

In [ ]:
data_combined = data_plasma_filtered_log_imputed_corrected.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)

In [ ]:
mask = data_plasma_filtered_log.isna()
data_reverted = data_plasma_filtered_log_imputed_corrected.mask(mask)
data_combined_noimpute = data_reverted.T.join(data_cli_prot).rename_axis('Sample ID', axis=0)
data_combined_noimpute = data_combined_noimpute.join(annotation_file.set_index('Sample ID'), how='left')

In [ ]:
#protein_ids_filtered = protein_ids.set_index('ProteinID_Genename').loc[data_plasma_filtered.index]
double_key = data_cli_prot[['blood_sample_number']]
annotation_file_export = annotation_file.set_index('Sample ID').join(double_key, how='right')

### Data exploration

#### Normality test before transformation

In [ ]:
# Long data format to ease computation 
data_long = data_combined[proteins].melt(var_name='ProteinID_Genename', value_name='MS signal [Log2]').set_index('ProteinID_Genename')

from src.statistical_testing import normality_pg
# Set a dummy variable needed for pingouin.normality
data_long['group_dummy']=1
normality_results = normality_pg(data=data_long, dv='MS signal [Log2]', group='group_dummy')

# Show results
print('Number of protein with non-normal distribution:{}'.format(normality_results['normal'].value_counts()[False]))

#### Ranked-based inverse normalized transformation (INT)
- https://stackoverflow.com/questions/15549836/transform-data-to-fit-normal-distribution
- [good discussion on violating OLS residual normality assumption](https://stats.stackexchange.com/questions/29731/regression-when-the-ols-residuals-are-not-normally-distributed)
- [good discussion on testing non-linear association](https://stats.stackexchange.com/questions/35893/how-do-i-test-a-nonlinear-association)

In [ ]:
# Perform ranked-based inverse normalized transformation (INT) on protein levels per protein
# Refer to https://github.com/edm1/rank-based-INT

from src.rank_based_int import rank_INT
RE_INT = False

if not RE_INT:
    data_proteomics_int = pd.read_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected_int.csv').set_index('Sample ID')
else:
    new_df = []
    for protein in tqdm(proteins):
        new_df.append(pd.DataFrame(rank_INT(data_plasma_filtered_log_imputed_corrected.T[protein], stochastic=False), columns=[protein]))
    data_proteomics_int = pd.concat(new_df, axis=1)
    data_proteomics_int.rename_axis('Sample ID', axis=0, inplace=True)
    data_proteomics_int.to_csv(DATA_FOLDER_PROCESSED / 'proteomics_filtered_imputed_corrected_int.csv')    

In [ ]:
value = data_proteomics_int.loc['PlateR1_2', 'Q9Y6Z7_COLEC10']
assert abs(value - -1.948646) < 1e-6, 'Normalized value changed in comparison to previous run'

In [ ]:
data_combined_int = data_proteomics_int.join(data_cli_prot).rename_axis('Sample ID', axis=0)
data_combined_int = data_combined_int.join(df_pc['PC1'], how='left')

#### Save dataset

In [ ]:
FILE_RESULTS = DATA_FOLDER_PROCESSED / 'proteomics_datasets.xlsx'

def format_dataset(df):
    df_new=df.copy()
    df_new.rename_axis('ProteinID_Genename', axis=0, inplace=True)
    cols_to_add = ['Protein ID', 'Protein IDs', 'Gene name', 'Gene names']
    df_new=df_new.join(protein_ids.set_index('ProteinID_Genename')[cols_to_add])
    return df_new

with pd.ExcelWriter(FILE_RESULTS) as writer:
    format_dataset(data_plasma_raw).to_excel(writer, sheet_name='raw')
    format_dataset(data_plasma_filtered_log).to_excel(writer, sheet_name='filtered_log2')
    format_dataset(data_plasma_filtered_log_imputed).to_excel(writer, sheet_name='filtered_log2_imputed')
    format_dataset(data_plasma_filtered_log_imputed_corrected).to_excel(writer, sheet_name='imputed_batchcorrected')
    format_dataset(data_reverted).to_excel(writer, sheet_name='batchcorrected_no_imputation')
    annotation_file_export.to_excel(writer, sheet_name='annotation_file')

### Data processing for protein-genotype association

In [ ]:
# overweight/obesity (BMI SDS >= 1.28)
data_combined_int['overweight'] = np.where(data_combined_int['z_BMI.Nysom']>=1.28, 1, 0)
data_combined_int['overweight*z_BMI.Nysom']=data_combined_int['overweight']*data_combined_int['z_BMI.Nysom']

In [ ]:
# Define the list of covariates for pqtl analysis
covariates_pqtl = ['age', 'sex', 'z_BMI.Nysom', 'overweight*z_BMI.Nysom', 
                   'overweight', 'time_to_analysis', 'PC1']

In [ ]:
RE_LIREG = False

if not RE_LIREG:
    stats_pqtl = pd.read_csv(gemma_path / 'lireg_statistics.csv')
    residuals_pqtl = pd.read_csv(gemma_path / 'lireg_residuals.csv').set_index('Sample ID')
else:
    stats_pqtl, residuals_pqtl = perform_linear_regression(data_combined_int, proteins, covariates_pqtl)    
    # Save results
    stats_pqtl.to_csv(gemma_path / 'lireg_statistics.csv', index=False)
    residuals_pqtl = pd.DataFrame.from_dict(residuals_pqtl).rename_axis('Sample ID', axis=0)
    residuals_pqtl.to_csv(gemma_path / 'lireg_residuals.csv')

In [ ]:
# Check if regression results changed in comparison to previous run
value = residuals_pqtl.loc['PlateR1_1', 'Q9Y6Z7_COLEC10']
assert abs(value - -0.424027) < 1e-6, 'Regressed value for GWAS changed in comparison to previous run'

#### Perform INT on residuals

In [ ]:
RE_INT = False

if not RE_INT:
    data_gwas_int = pd.read_csv(gemma_path / 'lireg_residuals_int.csv').set_index('Sample ID')
else:
    new_df = []
    data = residuals_pqtl
    for protein in tqdm(data.columns):
        new_df.append(pd.DataFrame(rank_INT(data[protein], stochastic=False), columns=[protein]))
    data_gwas_int = pd.concat(new_df, axis=1)
    data_gwas_int.rename_axis('Sample ID', axis=0, inplace=True)
    data_gwas_int.to_csv(gemma_path / 'lireg_residuals_int.csv')

#### Prepare the data to fit the GEMMA input format

In [ ]:
# Replace sample ID with participant ID so it's compatible with genotype data
participant_ids = '66-' + data_gwas_int.index.map(IDmapping_SampleID_to_bloodSampleID)
data_gwas_int.insert(0, 'Participant ID', participant_ids)

In [ ]:
# Import the .fam file template
fam_tep = pd.read_csv(folders['pQTL'] / 'QC_PLINK/target1k5.fam', header=None, sep=' ')
data_gwas_export = fam_tep.set_index(0).join(data_gwas_int.round(5).set_index('Participant ID')).reset_index().drop([5], axis=1)
data_gwas_export = data_gwas_export.replace(np.nan, 'NA')

# Check if data changed in comparison to previous run
value = data_gwas_export.loc[0, 'P01009_SERPINA1']
assert abs(value - 1.22515) < 1e-6, 'Exported value for GWAS changed in comparison to previous run'

# Export data for GWAS
data_gwas_export.to_csv(gemma_path / 'phenotype.fam', header=None, index=False, sep=' ')

In [ ]:
# Create a dataframe mapping phenotype IDs to protein IDs
nr_proteins = len(proteins)
proteinID_pqtl=pd.DataFrame({'Phenotype ID':np.arange(1, nr_proteins+1),'Protein ID':data_gwas_export.columns[5:]})
proteinID_pqtl.to_csv(gemma_path / 'ProteinID.txt', index=False, sep='\t')

with open(gemma_path / 'phenotype.list', 'w') as file:
    for line in np.arange(1, nr_proteins+2):
        file.write(str(line) + '\n')
        
# Association performed in Computerome (GEMMA v0.98.3)

#### Export data to look at "dose-response"

In [ ]:
# Define paths
dose_resp_path = folders['pQTL'] / 'dose-response'
dose_resp_path.mkdir(exist_ok=True)
excel_path = dose_resp_path / 'dose_response.xlsx'

# Define a function to prepare dataframes
def prepare_df(df, col_prefix, index_col=None):
    df = df.copy()
    if index_col:
        df[index_col] = col_prefix + df.index.map(IDmapping_SampleID_to_bloodSampleID)
        df = df.set_index(index_col)
    else:
        df.columns = col_prefix + df.columns.map(IDmapping_SampleID_to_bloodSampleID)
    return df

# Writing to Excel with prepared dataframes
with pd.ExcelWriter(excel_path) as writer:
    prepare_df(data_plasma_filtered_log_imputed_corrected, '66-').to_excel(writer, sheet_name='data_log2_imputed')
    prepare_df(data_reverted, '66-').to_excel(writer, sheet_name='data_log2')
    prepare_df(data_cli_prot[['sex', 'age_year', 'obesity']].dropna(), '66-', 'Participant ID').to_excel(writer, sheet_name='data_cli')

#### Export sample ID annotation for peptide data inspection

In [ ]:
df_annotation_pep = annotation_file.copy()
df_annotation_pep['Participant ID'] = '66-' + df_annotation_pep['Sample ID'].map(IDmapping_SampleID_to_bloodSampleID)
df_annotation_pep.to_csv(folders['pQTL'] / 'peptide_annotation.csv')

# Peptide-level inspection was performed in Computerome

#### Prepare genotype array batch data for use as covariates ####

In [ ]:
# import batch info
df_batch = pd.read_csv(folders['pQTL'] / 'ids_in_batch.txt', sep=' ', header=None, names=['participant ID', 'batch'])
onehot_batch = pd.DataFrame(pd.get_dummies(df_batch['batch']))
onehot_batch['participant ID']=df_batch['participant ID']

In [ ]:
# Combine participant IDs and genotype array batch data as covariates
data_gwas_covariates = fam_tep[[0]].set_index(0).join(onehot_batch.set_index('participant ID')).drop(['batch2015'], axis=1)
# Add an intercept column to the covariate data
data_gwas_covariates.insert(0, 'intercept', 1)
# Replace missing values with 'NA'
data_gwas_covariates = data_gwas_covariates.fillna('NA')
# Save the covariate data to a tab-separated text file for use with GEMMA
data_gwas_covariates.to_csv(gemma_path / 'covariates.txt', header=None, index=False, sep=' ')

#### Check sex discrepancy

In [ ]:
# Read in the PLINK sexcheck file and merge with clinical data
df_sex = pd.read_csv(folders['pQTL'] / 'QC_PLINK/plink.sexcheck', sep='\s+')
df_sex['biobank_ID']=df_sex['FID'].str.split('-').str[1]
df_sex = df_sex.merge(data_cli_raw[['biobank_ID', 'sex']], on='biobank_ID', how='left')

# Select rows where the SNPSEX column does not match the sex column from clinical data
df_sex[df_sex['SNPSEX']!=df_sex['sex']]